In [1]:
!pip -q install altair vl-convert-python Pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 45.1 MB/s eta 0:00:00


In [2]:
import pandas as pd
import altair as alt
from PIL import Image

CSV_NAME = "cohorte_2019.csv"

# 1. Cargar datos
df = pd.read_csv(CSV_NAME)
df["anio_nac"] = pd.to_datetime(df["fecha_nacimiento"]).dt.year
df["regla"] = df["anio_nac"].apply(
    lambda y: "Nacido en 1999 o después" if y >= 1999 else "Nacido antes de 1999 (revisar)"
)

# 2. Configurar visualización
ESTATUS = ["Consolidado Primera Division", "Transferido al Extranjero", "Ascenso / Segunda"]
COLORES = ["#0072B2", "#009E73", "#D55E00"]
REGLA = ["Nacido en 1999 o después", "Nacido antes de 1999 (revisar)"]
DESTACADOS = ["Ignacio Saavedra", "Tomas Alarcon", "Carlos Palacios", "Vicente Fernandez"]

sel = alt.selection_point(fields=["estatus_carrera_24"], bind="legend")

puntos = (
    alt.Chart(df)
    .mark_point(filled=True, stroke="white", strokeWidth=0.8)
    .encode(
        x=alt.X("minutos_liga_2019:Q", title="Minutos en liga 2019 (cohorte Sub-21)",
                scale=alt.Scale(domain=[0, 2000])),
        y=alt.Y("minutos_liga_edad_24:Q", title="Minutos en liga en el año en que cumplió 24",
                scale=alt.Scale(domain=[0, 2600])),
        color=alt.Color("estatus_carrera_24:N", title="Estatus a los 24 (clic para filtrar)",
                        scale=alt.Scale(domain=ESTATUS, range=COLORES)),
        size=alt.Size("minutos_liga_edad_22:Q", title="Minutos a los 22",
                      scale=alt.Scale(range=[60, 900])),
        shape=alt.Shape("regla:N", title="Criterio de cohorte",
                        scale=alt.Scale(domain=REGLA, range=["circle", "diamond"])),
        opacity=alt.condition(sel, alt.value(0.9), alt.value(0.15)),
        tooltip=[
            alt.Tooltip("nombre_jugador:N", title="Jugador"),
            alt.Tooltip("club_2019:N", title="Club 2019"),
            alt.Tooltip("posicion_especifica:N", title="Posición"),
            alt.Tooltip("minutos_liga_2019:Q", title="Min. 2019", format=","),
            alt.Tooltip("minutos_liga_edad_22:Q", title="Min. a los 22", format=","),
            alt.Tooltip("club_edad_22:N", title="Club a los 22"),
            alt.Tooltip("minutos_liga_edad_24:Q", title="Min. a los 24", format=","),
            alt.Tooltip("club_edad_24:N", title="Club a los 24"),
            alt.Tooltip("ratio_retencion_22:Q", title="Retención a los 22 (%)"),
            alt.Tooltip("estatus_carrera_24:N", title="Estatus a los 24"),
        ],
    )
    .add_params(sel)
    .interactive()
)

diagonal = (
    alt.Chart(pd.DataFrame({"x": [0, 2000], "y": [0, 2000]}))
    .mark_line(strokeDash=[5, 4], color="#888888")
    .encode(x="x:Q", y="y:Q")
)

etiquetas = (
    alt.Chart(df)
    .transform_filter(alt.FieldOneOfPredicate(field="nombre_jugador", oneOf=DESTACADOS))
    .mark_text(align="left", dx=10, dy=-8, fontSize=11)
    .encode(x="minutos_liga_2019:Q", y="minutos_liga_edad_24:Q", text="nombre_jugador:N")
)

grafico = (
    (diagonal + puntos + etiquetas)
    .properties(
        width=720, height=480,
        title=alt.TitleParams(
            "¿Los minutos Sub-21 de 2019 anticipan la carrera a los 24 años?",
            subtitle=["Cada punto es un jugador. Sobre la línea punteada: más minutos a los 24 que en 2019.",
                      "Tamaño = minutos a los 22. Base provisional, en verificación."],
            anchor="start",
        ),
    )
    .configure_axis(grid=True, gridColor="#eeeeee", labelFontSize=11, titleFontSize=12)
    .configure_legend(labelFontSize=11, titleFontSize=11)
)

# 3. Exportar HTML e Imagen
grafico.save("vis_01.html")
grafico.save("vis_01.png", scale_factor=2)
Image.open("vis_01.png").convert("RGB").save("vis_01.jpg", quality=95)

print("¡Listo! vis_01.html y vis_01.jpg generados con éxito.")
grafico

¡Listo! vis_01.html y vis_01.jpg generados con éxito.


alt.LayerChart(...)